In [1]:
import torch
import torch.nn as nn
import bidirectional_dataset
import two_way_seq2seq
import trainer
import plotting
import importlib
import neptune

importlib.reload(bidirectional_dataset)
importlib.reload(trainer)
importlib.reload(plotting)


/home/dl11e23/.conda-2022.05/envs/sleep/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'plotting' from '/home/dl11e23/sleep-time-series/plotting.py'>

# Setup hyperparameters

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Exclude subject 006
subjects = ["001", "002", "003", "004", "005", "007", "008", "009"]

# Use all data
training_proportion_to_use = 1
validation_proportion_to_use = 1

# in seconds
time_before_cutout = 1
cutout_duration = 1
time_after_cutout = 1

# defining the resampling stuff
original_freq = 200
resample_freq = 20



# HYPERPARAMETERS

batch_size = 256
hidden_size = 1024
lr = 0.001
initial_teacher_forcing = 0.2
encoder_dropout = 0.2
num_layers = 3
n_epoch = 60
epochs_with_teacher_forcing = 3

n_channels = 7
cutout_duration_steps = cutout_duration * resample_freq

# Cross validation loop 
For each subject, train the model on all other subjects and validate on given subject. 

In [ ]:
for i in range(len(subjects)): # Loop through all subjects
    # Start neptune run
    run = neptune.init_run(
    project="sleep-time-series/sleep-time-series",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJlNjYwOTA1Zi02ZDhiLTQ5NGYtODUwYy1jNzA3ZmM0MjBhMWEifQ==",
    name=f"{subjects[i]}",
    tags=["cross validation"]
    )
    
    # Save parameters to neptune
    run["params"] = {
    "batch_size": batch_size,
    "hidden_size": hidden_size,
    "learning_rate": lr,
    "initial_teacher_forcing": initial_teacher_forcing,
    "encoder_droput": encoder_dropout,
    "n_epoch": n_epoch,
    "epochs_with_teacher_forcing": epochs_with_teacher_forcing,
    "num_layers": num_layers
    }
    
    training_subjects = subjects[:i] + subjects[i+1:] # all other subjects is training data
    validation_subjects = [subjects[i]] # one subject is validation data
    
    # Save training and validation subjects to neptune
    run["training_subjects"] = ", ".join(training_subjects)
    run["validation_subjects"] = ", ".join(validation_subjects)
    
    # create datasets
    combined_train_dataset, individual_train_datasets, standardization_info = bidirectional_dataset.create_combined_dataset(subjects=training_subjects,
                                                                                proportion_to_use=1, time_before_cutout=time_before_cutout,
                                                                                cutout_duration=cutout_duration, resample_freq=resample_freq)
    
    # Create a combined and individual validation dataset
    combined_validation_dataset, individual_validation_datasets, _ = bidirectional_dataset.create_combined_dataset(subjects=validation_subjects,
                                                                                proportion_to_use=1, time_before_cutout=time_before_cutout,
                                                                                cutout_duration=cutout_duration, resample_freq=resample_freq,
                                                                                standardization_info=standardization_info)
    
    # Create dataloaders
    train_loader = torch.utils.data.DataLoader(combined_train_dataset, batch_size=batch_size, shuffle=True)
    validation_loader = torch.utils.data.DataLoader(combined_validation_dataset, batch_size=batch_size, shuffle=False)
    
    # Setup the model
    pred_model = two_way_seq2seq.TwoWaySeq2Seq(hidden_size, n_channels, cutout_duration_steps, encoder_dropout=encoder_dropout, num_layers=num_layers)
    pred_model = nn.DataParallel(pred_model, device_ids = [0, 1])
    pred_model.to(device)
    optimizer = torch.optim.Adam(params = pred_model.parameters(), lr = lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience = 10, factor=0.5)
    loss_fn = nn.MSELoss(reduction="mean")

    
    # Train the model
    trainer.pred_training_loop(pred_model, n_epoch, loss_fn, optimizer, train_loader, validation_loader, 
                                    neptune_run=run,
                                    device=device, scheduler=scheduler, 
                                    initial_teacher_forcing=initial_teacher_forcing, 
                                    epochs_with_teacher_forcing=epochs_with_teacher_forcing)
    
    # Plots for neptune
    plot1 = plotting.plot_target_and_prediction(pred_model, individual_validation_datasets, 40, 0, device,
                                   time_before_cutout=time_before_cutout, cutout_duration=cutout_duration,
                                   time_after_cutout=time_after_cutout)
    plot2 = plotting.plot_target_and_prediction(pred_model, individual_validation_datasets, 200, 0, device,
                                   time_before_cutout=time_before_cutout, cutout_duration=cutout_duration,
                                   time_after_cutout=time_after_cutout)
    plot3 = plotting.plot_target_and_prediction(pred_model, individual_validation_datasets, 600, 0, device,
                                   time_before_cutout=time_before_cutout, cutout_duration=cutout_duration,
                                   time_after_cutout=time_after_cutout)
    # Save plots to neptune 
    run["plot1"].upload(plot1)
    run["plot2"].upload(plot2)
    run["plot3"].upload(plot3)
    run.stop()

/tmp/ipykernel_1762807/176693329.py:2: NeptuneWarning: The following monitoring options are disabled by default in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', and 'capture_hardware_metrics'. To enable them, set each parameter to 'True' when initializing the run. The monitoring will continue until you call run.stop() or the kernel stops. Also note: Your source files can only be tracked if you pass the path(s) to the 'source_code' argument. For help, see the Neptune docs: https://docs.neptune.ai/logging/source_code/
  run = neptune.init_run(


https://app.neptune.ai/sleep-time-series/sleep-time-series/e/SLEEP-410


/home/dl11e23/sleep-time-series/bidirectional_dataset.py:152: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = read_raw_bids(bids_path=bids_path, verbose=False)
/home/dl11e23/sleep-time-series/bidirectional_dataset.py:152: RuntimeWarning: Expected to find a single events.tsv file associated with sub-002_ses-001_task-sleep, but found 2:

data/eesm17/sub-002/ses-001/eeg/sub-002_ses-001_task-sleep_events.tsv
data/eesm17/sub-002/ses-001/eeg/sub-002_ses-001_task-sleep_acq-scoring_events.tsv

The search_str was "data/eesm17/sub-002/**/eeg/sub-002_ses-001*events.tsv"
  raw = read_raw_bids(bids_path=bids_path, verbose=False)
/home/dl11e23/sleep-time-series/bidirectional_dataset.py:152: RuntimeWarning: The number of channels in the channels.tsv sidecar file (34) does not match the number of channels in the raw data file (35). Will not try to set channel names.
  raw = read_raw_bids(bids_path=bids_path, ve

resampling...
Loading data for subject 002


/home/dl11e23/sleep-time-series/bidirectional_dataset.py:152: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = read_raw_bids(bids_path=bids_path, verbose=False)
/home/dl11e23/sleep-time-series/bidirectional_dataset.py:152: RuntimeWarning: Expected to find a single events.tsv file associated with sub-003_ses-001_task-sleep, but found 2:

data/eesm17/sub-003/ses-001/eeg/sub-003_ses-001_task-sleep_acq-scoring_events.tsv
data/eesm17/sub-003/ses-001/eeg/sub-003_ses-001_task-sleep_events.tsv

The search_str was "data/eesm17/sub-003/**/eeg/sub-003_ses-001*events.tsv"
  raw = read_raw_bids(bids_path=bids_path, verbose=False)
/home/dl11e23/sleep-time-series/bidirectional_dataset.py:152: RuntimeWarning: The number of channels in the channels.tsv sidecar file (34) does not match the number of channels in the raw data file (35). Will not try to set channel names.
  raw = read_raw_bids(bids_path=bids_path, ve

...

resampling...
Loading data for subject 008
Reading 0 ... 2395199  =      0.000 ... 11975.995 secs...


/home/dl11e23/sleep-time-series/bidirectional_dataset.py:152: RuntimeWarning: Expected to find a single events.tsv file associated with sub-009_ses-001_task-sleep, but found 2:

data/eesm17/sub-009/ses-001/eeg/sub-009_ses-001_task-sleep_events.tsv
data/eesm17/sub-009/ses-001/eeg/sub-009_ses-001_task-sleep_acq-scoring_events.tsv

The search_str was "data/eesm17/sub-009/**/eeg/sub-009_ses-001*events.tsv"
  raw = read_raw_bids(bids_path=bids_path, verbose=False)


resampling...
Loading data for subject 009
Reading 0 ... 5557599  =      0.000 ... 27787.995 secs...


/home/dl11e23/sleep-time-series/bidirectional_dataset.py:152: RuntimeWarning: Expected to find a single events.tsv file associated with sub-001_ses-001_task-sleep, but found 2:

data/eesm17/sub-001/ses-001/eeg/sub-001_ses-001_task-sleep_events.tsv
data/eesm17/sub-001/ses-001/eeg/sub-001_ses-001_task-sleep_acq-scoring_events.tsv

The search_str was "data/eesm17/sub-001/**/eeg/sub-001_ses-001*events.tsv"
  raw = read_raw_bids(bids_path=bids_path, verbose=False)


resampling...
Loading data for subject 001
